In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_schemaorg.jsonld", "data/metadata/climate_dataset_schemaorg.jsonld"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Downloaded: data/metadata/climate_dataset_schemaorg.jsonld
Bootstrap complete.


# Day 2: Processing schema.org Metadata Step by Step

In this notebook, we study **schema.org JSON-LD** metadata for a dataset and process it gradually.

## Learning goals
By the end of this notebook, students should be able to:

1. understand what a schema.org metadata record looks like,
2. identify the most important fields for a dataset,
3. extract key metadata with Python,
4. convert nested JSON-LD into a simpler table,
5. relate the extracted metadata to **FAIR**, especially **Findable** and **Reusable**.

## Dataset used
We use the metadata file for:

**2025 Global Climate Data**

This is a teaching example used throughout Week 1 so that students can compare:
- **Dublin Core**
- **DataCite**
- **schema.org**

in a consistent way.


## Step 1 — Load the JSON-LD file

We first locate the local schema.org metadata file that was downloaded by the bootstrap cell.


In [2]:
from pathlib import Path
import json
import pandas as pd

schema_path = Path("data/metadata/climate_dataset_schemaorg.jsonld")

print("Schema.org file path:", schema_path)
print("File exists:", schema_path.exists())

if not schema_path.exists():
    raise FileNotFoundError(f"Could not find: {schema_path}")

Schema.org file path: data/metadata/climate_dataset_schemaorg.jsonld
File exists: True


## Step 2 — Read the raw metadata file

Before parsing, it is useful to inspect the raw JSON-LD structure directly.


In [3]:
raw_text = schema_path.read_text(encoding="utf-8")
print(raw_text[:2000])

{
  "@context": "https://schema.org",
  "@type": "Dataset",
  "name": "2025 Global Climate Data",
  "description": "A small teaching dataset containing fictional annual climate summaries. Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.",
  "creator": [
    {
      "@type": "Organization",
      "name": "Global Climate Data Team"
    }
  ],
  "publisher": {
    "@type": "Organization",
    "name": "Open Research Repository (Teaching)"
  },
  "identifier": "10.1234/gcd-2025",
  "keywords": [
    "climate",
    "temperature",
    "rainfall",
    "CO2 emissions",
    "teaching dataset"
  ],
  "license": "https://creativecommons.org/licenses/by/4.0/",
  "distribution": [
    {
      "@type": "DataDownload",
      "encodingFormat": "text/csv",
      "contentUrl": "data/raw/climate_sample.csv",
      "name": "climate_sample.csv"
    }
  ]
}




## Step 3 — Load the JSON-LD object into Python

Now we read the JSON-LD content into a Python dictionary.


In [4]:
obj = json.loads(raw_text)

print("Python object type:", type(obj))
if isinstance(obj, dict):
    print("Top-level keys:", list(obj.keys()))
else:
    print("Top-level object is not a dictionary.")

Python object type: <class 'dict'>
Top-level keys: ['@context', '@type', 'name', 'description', 'creator', 'publisher', 'identifier', 'keywords', 'license', 'distribution']


## Step 4 — Identify the dataset record

Sometimes JSON-LD is a single dictionary.  
Sometimes it is stored inside **`@graph`** with multiple objects.

For teaching, we explicitly show how to find the record whose `@type` is `Dataset`.


In [5]:
if isinstance(obj, dict) and "@graph" in obj and isinstance(obj["@graph"], list):
    dataset_record = next(
        (item for item in obj["@graph"] if isinstance(item, dict) and item.get("@type") == "Dataset"),
        None
    )
else:
    dataset_record = obj if isinstance(obj, dict) else None

print("Dataset record found:", dataset_record is not None)
if dataset_record:
    print("Dataset record keys:")
    print(list(dataset_record.keys()))

Dataset record found: True
Dataset record keys:
['@context', '@type', 'name', 'description', 'creator', 'publisher', 'identifier', 'keywords', 'license', 'distribution']


## Step 5 — Inspect important schema.org fields

For a dataset, we are especially interested in fields such as:
- `@context`
- `@type`
- `name`
- `description`
- `creator`
- `identifier`
- `keywords`
- `license`
- `distribution`
- `publisher`

These fields help students connect metadata to **discovery**, **understanding**, and **reuse**.


In [6]:
important_fields = [
    "@context", "@type", "name", "description", "creator",
    "identifier", "keywords", "license", "distribution", "publisher"
]

for field in important_fields:
    print(f"\n--- {field} ---")
    value = dataset_record.get(field) if dataset_record else None
    print(value)


--- @context ---
https://schema.org

--- @type ---
Dataset

--- name ---
2025 Global Climate Data

--- description ---
A small teaching dataset containing fictional annual climate summaries. Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.

--- creator ---
[{'@type': 'Organization', 'name': 'Global Climate Data Team'}]

--- identifier ---
10.1234/gcd-2025

--- keywords ---
['climate', 'temperature', 'rainfall', 'CO2 emissions', 'teaching dataset']

--- license ---
https://creativecommons.org/licenses/by/4.0/

--- distribution ---
[{'@type': 'DataDownload', 'encodingFormat': 'text/csv', 'contentUrl': 'data/raw/climate_sample.csv', 'name': 'climate_sample.csv'}]

--- publisher ---
{'@type': 'Organization', 'name': 'Open Research Repository (Teaching)'}


## Step 6 — Create a clean parser function

Next, we write a small function that extracts the most useful fields in a classroom-friendly format.

This step is important because raw JSON-LD may be nested, while students usually need a clean summary table.


In [9]:
def parse_schemaorg_jsonld(jsonld_path):
    """Parse a local schema.org JSON-LD file and return a simplified metadata dictionary."""
    obj = json.loads(Path(jsonld_path).read_text(encoding="utf-8"))

    # Handle either a direct Dataset object or a Dataset inside @graph
    ds = obj
    if isinstance(obj, dict) and "@graph" in obj and isinstance(obj["@graph"], list):
        ds = next(
            (x for x in obj["@graph"] if isinstance(x, dict) and x.get("@type") == "Dataset"),
            None
        )

    if not isinstance(ds, dict):
        raise ValueError("No valid Dataset object found in the JSON-LD file.")

    # Creator extraction
    creator_value = ds.get("creator", [])
    creators = []
    if isinstance(creator_value, list):
        for c in creator_value:
            if isinstance(c, dict):
                name = c.get("name")
                if name:
                    creators.append(name)
            elif c:
                creators.append(str(c))
    elif isinstance(creator_value, dict):
        name = creator_value.get("name")
        if name:
            creators.append(name)
    elif creator_value:
        creators.append(str(creator_value))

    # Keyword extraction
    keywords_value = ds.get("keywords", [])
    if isinstance(keywords_value, str):
        keywords = [k.strip() for k in keywords_value.split(",") if k.strip()]
    elif isinstance(keywords_value, list):
        keywords = [str(k).strip() for k in keywords_value if str(k).strip()]
    else:
        keywords = []

    # Publisher extraction
    publisher_value = ds.get("publisher")
    if isinstance(publisher_value, dict):
        publisher = publisher_value.get("name")
    else:
        publisher = publisher_value

    # Distribution / encoding format extraction
    distribution_value = ds.get("distribution")
    file_format = None
    if isinstance(distribution_value, dict):
        file_format = distribution_value.get("encodingFormat")
    elif isinstance(distribution_value, list) and distribution_value:
        first = distribution_value[0]
        if isinstance(first, dict):
            file_format = first.get("encodingFormat")

    return {
        "type": ds.get("@type"),
        "title": ds.get("name"),
        "creator": creators,
        "identifier": ds.get("identifier"),
        "publisher": publisher,
        "description": ds.get("description"),
        "keywords": keywords,
        "license": ds.get("license"),
        "format": file_format,
    }

## Step 7 — Parse the metadata record

Now we run the parser and inspect the simplified output.


In [8]:
meta = parse_schemaorg_jsonld(schema_path)
meta

{'type': 'Dataset',
 'title': '2025 Global Climate Data',
 'creator': ['Global Climate Data Team'],
 'identifier': '10.1234/gcd-2025',
 'publisher': 'Open Research Repository (Teaching)',
 'description': 'A small teaching dataset containing fictional annual climate summaries. Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.',
 'keywords': ['climate',
  'temperature',
  'rainfall',
  'CO2 emissions',
  'teaching dataset'],
 'license': 'https://creativecommons.org/licenses/by/4.0/',
 'format': 'text/csv'}

## Step 8 — Convert the parsed metadata into a table

A table is much easier to discuss in class than a nested JSON structure.


In [10]:
flattened = {
    "@type": meta.get("type"),
    "title": meta.get("title"),
    "creator": "; ".join(meta.get("creator") or []),
    "identifier": meta.get("identifier"),
    "publisher": meta.get("publisher"),
    "description": meta.get("description"),
    "keywords": "; ".join(meta.get("keywords") or []),
    "license": meta.get("license"),
    "format": meta.get("format"),
}

df = pd.DataFrame(list(flattened.items()), columns=["field", "value"])
df

,field,value
0,@type,Dataset
1,title,2025 Global Climate Data
2,creator,Global Climate Data Team
3,identifier,10.1234/gcd-2025
4,publisher,Open Research Repository (Teaching)
5,description,A small teaching dataset containing fictional ...
6,keywords,climate; temperature; rainfall; CO2 emissions;...
7,license,https://creativecommons.org/licenses/by/4.0/
8,format,text/csv


## Step 9 — Highlight metadata that supports FAIR

Now we interpret the extracted fields through the FAIR lens.

### Fields that support **Findable**
- `identifier`
- `title`
- `keywords`
- `@type`

### Fields that support **Reusable**
- `description`
- `license`
- `format`
- `publisher`

These are not the full FAIR criteria, but they are strong indicators that students can inspect easily.


In [11]:
fair_view = pd.DataFrame([
    {"FAIR principle": "Findable", "Relevant fields": "identifier, title, keywords, @type"},
    {"FAIR principle": "Reusable", "Relevant fields": "description, license, format, publisher"},
])
fair_view

,FAIR principle,Relevant fields
0,Findable,"identifier, title, keywords, @type"
1,Reusable,"description, license, format, publisher"


## Step 10 — Check metadata completeness

This is a simple teaching-oriented check, not a formal validator.

We inspect whether key fields are present.


In [12]:
checks = {
    "Has title": bool(meta.get("title")),
    "Has creator": bool(meta.get("creator")),
    "Has identifier": bool(meta.get("identifier")),
    "Has description": bool(meta.get("description")),
    "Has keywords": bool(meta.get("keywords")),
    "Has license": bool(meta.get("license")),
    "Has format": bool(meta.get("format")),
}

checks_df = pd.DataFrame(list(checks.items()), columns=["check", "result"])
checks_df

,check,result
0,Has title,True
1,Has creator,True
2,Has identifier,True
3,Has description,True
4,Has keywords,True
5,Has license,True
6,Has format,True


## Step 11 — Weaknesses and discussion points

Even if a schema.org record looks good, students should still ask:

- Is the **identifier** persistent and globally unique?
- Are the **keywords** specific enough for discovery?
- Is the **description** detailed enough for reuse?
- Is the **license** clear?
- Is the **format** sufficient for someone else to use the dataset?
- Are there missing fields such as temporal or spatial coverage?

These questions help students move from **reading metadata** to **evaluating metadata quality**.


In [13]:
discussion_points = pd.DataFrame([
    {"Question": "Is the identifier clear and stable?", "Why it matters": "Supports Findability"},
    {"Question": "Are the keywords informative?", "Why it matters": "Improves search and discovery"},
    {"Question": "Is the description sufficiently detailed?", "Why it matters": "Supports Reusability"},
    {"Question": "Is the license explicitly stated?", "Why it matters": "Clarifies reuse conditions"},
    {"Question": "Is the file format described?", "Why it matters": "Helps technical reuse"},
])
discussion_points

,Question,Why it matters
0,Is the identifier clear and stable?,Supports Findability
1,Are the keywords informative?,Improves search and discovery
2,Is the description sufficiently detailed?,Supports Reusability
3,Is the license explicitly stated?,Clarifies reuse conditions
4,Is the file format described?,Helps technical reuse


## Final takeaway

In this notebook, we moved step by step from:
1. raw JSON-LD,
2. to Python object inspection,
3. to extraction of important metadata fields,
4. to a clean classroom table,
5. to FAIR-oriented interpretation.

This makes schema.org less abstract and helps students see metadata as something practical, inspectable, and useful.
